In [1]:
import requests
import pandas as pd
from datetime import datetime
import os

In [2]:
# GitHub API base URL
BASE_URL = "https://api.github.com/search/repositories"

# Languages we want to track
LANGUAGES = [
    "Python", "JavaScript", "TypeScript", "Rust", "Go",
    "Java", "C++", "Kotlin", "Swift", "Ruby"
]

# Where to save the data
OUTPUT_FILE = "../data/snapshots.csv"

In [3]:
def fetch_language_stats(language):
    """
    Fetch repo stats for a given language from GitHub API.
    Returns a dictionary with the stats we care about.
    """
    
    # Search for repos created in the last 7 days for this language
    query = f"language:{language} created:>2024-01-01 stars:>10"
    
    params = {
        "q": query,
        "sort": "stars",
        "order": "desc",
        "per_page": 10
    }
    
    headers = {
        "Accept": "application/vnd.github.v3+json"
    }
    
    response = requests.get(BASE_URL, params=params, headers=headers)
    
    if response.status_code != 200:
        print(f"Error fetching {language}: {response.status_code}")
        return None
    
    data = response.json()
    
    return {
        "language": language,
        "total_repos": data["total_count"],
        "top_repo_name": data["items"][0]["full_name"] if data["items"] else None,
        "top_repo_stars": data["items"][0]["stargazers_count"] if data["items"] else 0,
        "top_repo_url": data["items"][0]["html_url"] if data["items"] else None,
        "top_repo_description": data["items"][0]["description"] if data["items"] else None,
        "snapshot_date": datetime.today().strftime("%Y-%m-%d")
    }

In [4]:
# Test with just Python first
test = fetch_language_stats("Python")
print(test)

{'language': 'Python', 'total_repos': 110355, 'top_repo_name': 'NousResearch/hermes-agent', 'top_repo_stars': 128012, 'top_repo_url': 'https://github.com/NousResearch/hermes-agent', 'top_repo_description': 'The agent that grows with you', 'snapshot_date': '2026-05-01'}


In [5]:
# Fetch stats for all languages
print("Fetching data from GitHub API...")

all_stats = []

for language in LANGUAGES:
    print(f"  Fetching {language}...")
    stats = fetch_language_stats(language)
    if stats:
        all_stats.append(stats)

print(f"\nDone! Got data for {len(all_stats)} languages.")

Fetching data from GitHub API...
  Fetching Python...
  Fetching JavaScript...
  Fetching TypeScript...
  Fetching Rust...
  Fetching Go...
  Fetching Java...
  Fetching C++...
  Fetching Kotlin...
  Fetching Swift...
  Fetching Ruby...

Done! Got data for 10 languages.


In [6]:
# Convert to a pandas DataFrame
df_today = pd.DataFrame(all_stats)

# Preview it
df_today

,language,total_repos,top_repo_name,top_repo_stars,top_repo_url,top_repo_description,snapshot_date
0,Python,110355,NousResearch/hermes-agent,128013,https://github.com/NousResearch/hermes-agent,The agent that grows with you,2026-05-01
1,JavaScript,29834,affaan-m/everything-claude-code,171432,https://github.com/affaan-m/everything-claude-...,The agent harness performance optimization sys...,2026-05-01
2,TypeScript,49258,openclaw/openclaw,367120,https://github.com/openclaw/openclaw,Your own personal AI assistant. Any OS. Any Pl...,2026-05-01
3,Rust,15995,ultraworkers/claw-code,189525,https://github.com/ultraworkers/claw-code,The repo is finally unlocked. enjoy the party!...,2026-05-01
4,Go,13569,danielmiessler/Fabric,41314,https://github.com/danielmiessler/Fabric,Fabric is an open-source framework for augment...,2026-05-01
5,Java,8522,opendataloader-project/opendataloader-pdf,19844,https://github.com/opendataloader-project/open...,PDF Parser for AI-ready data. Automate PDF acc...,2026-05-01
6,C++,14954,LadybirdBrowser/ladybird,62618,https://github.com/LadybirdBrowser/ladybird,Truly independent web browser,2026-05-01
7,Kotlin,5258,kavishdevar/librepods,26774,https://github.com/kavishdevar/librepods,AirPods liberated from Apple's ecosystem.,2026-05-01
8,Swift,4623,apple/container,26263,https://github.com/apple/container,A tool for creating and running Linux containe...,2026-05-01
9,Ruby,1208,antiwork/gumroad,8995,https://github.com/antiwork/gumroad,Sell stuff and see what sticks,2026-05-01


In [7]:
# Check if the CSV already exists
if os.path.exists(OUTPUT_FILE):
    # File exists — load it and append today's data
    df_existing = pd.read_csv(OUTPUT_FILE)
    df_combined = pd.concat([df_existing, df_today], ignore_index=True)
    print(f"Appended to existing file. Total rows: {len(df_combined)}")
else:
    # First time — just save today's data
    df_combined = df_today
    print(f"Created new file with {len(df_combined)} rows.")

# Save to CSV
df_combined.to_csv(OUTPUT_FILE, index=False)
print(f"Saved to {OUTPUT_FILE}")

Created new file with 10 rows.
Saved to ../data/snapshots.csv


In [8]:
# Read it back and confirm it looks right
df_check = pd.read_csv(OUTPUT_FILE)
print(f"Rows in file: {len(df_check)}")
print(f"Columns: {list(df_check.columns)}")
df_check

Rows in file: 10
Columns: ['language', 'total_repos', 'top_repo_name', 'top_repo_stars', 'top_repo_url', 'top_repo_description', 'snapshot_date']


,language,total_repos,top_repo_name,top_repo_stars,top_repo_url,top_repo_description,snapshot_date
0,Python,110355,NousResearch/hermes-agent,128013,https://github.com/NousResearch/hermes-agent,The agent that grows with you,2026-05-01
1,JavaScript,29834,affaan-m/everything-claude-code,171432,https://github.com/affaan-m/everything-claude-...,The agent harness performance optimization sys...,2026-05-01
2,TypeScript,49258,openclaw/openclaw,367120,https://github.com/openclaw/openclaw,Your own personal AI assistant. Any OS. Any Pl...,2026-05-01
3,Rust,15995,ultraworkers/claw-code,189525,https://github.com/ultraworkers/claw-code,The repo is finally unlocked. enjoy the party!...,2026-05-01
4,Go,13569,danielmiessler/Fabric,41314,https://github.com/danielmiessler/Fabric,Fabric is an open-source framework for augment...,2026-05-01
5,Java,8522,opendataloader-project/opendataloader-pdf,19844,https://github.com/opendataloader-project/open...,PDF Parser for AI-ready data. Automate PDF acc...,2026-05-01
6,C++,14954,LadybirdBrowser/ladybird,62618,https://github.com/LadybirdBrowser/ladybird,Truly independent web browser,2026-05-01
7,Kotlin,5258,kavishdevar/librepods,26774,https://github.com/kavishdevar/librepods,AirPods liberated from Apple's ecosystem.,2026-05-01
8,Swift,4623,apple/container,26263,https://github.com/apple/container,A tool for creating and running Linux containe...,2026-05-01
9,Ruby,1208,antiwork/gumroad,8995,https://github.com/antiwork/gumroad,Sell stuff and see what sticks,2026-05-01
